# Lekce 06 - Funkce

Obsah:

* proč funkce
* definice a volání
* parametry a argumenty
* signatura, anotace typů a docstring
* `return`, `None`, víc návratových hodnot
* co funkce vidí
* čistá funkce a vedlejší efekty
* funkce je hodnota
* jedna funkce na celou kolekci
* metody jsou taky funkce
* časté chyby

## 1. Proč funkce

Ve cvičení 2 jste počítali tepelný odpor vrstvy `R = d / lambda`. Pro tři vrstvy
je tentýž výpočet opsaný třikrát pod sebou:

In [1]:
d_zdivo = 0.30
lambda_zdivo = 0.18
R_zdivo = d_zdivo / lambda_zdivo

d_izolace = 0.16
lambda_izolace = 0.038
R_izolace = d_izolace / lambda_izolace

d_omitka = 0.015
lambda_omitka = 0.80
R_omitka = d_omitka / lambda_omitka

print(R_zdivo, R_izolace, R_omitka)

1.6666666666666667 4.2105263157894735 0.01875


Tři problémy: tentýž vzorec je tu třikrát, nikde není pojmenovaný, a kdyby byl
špatně, musíme ho opravit na třech místech. Funkce všechny tři vyřeší.

In [2]:
def odpor_vrstvy(d, lam):      # "lambda" je klicove slovo, proto lam
    return d / lam


R_zdivo = odpor_vrstvy(0.30, 0.18)
R_izolace = odpor_vrstvy(0.16, 0.038)
R_omitka = odpor_vrstvy(0.015, 0.80)

print(R_zdivo, R_izolace, R_omitka)

1.6666666666666667 4.2105263157894735 0.01875


Zásady, podle kterých se kód na funkce dělí:

* **jedna funkce, jedna úloha**. Když popis funkce obsahuje "a", je to nejspíš
  na dvě funkce
* **krátká**. Co se nevejde na obrazovku, patří rozdělit
* **jméno říká, co funkce dělá nebo co vrací**: `odpor_vrstvy`, `spocti_cenu`
* **všechno potřebné přijde argumenty**. Funkce, která si hodnoty bere odjinud,
  se nedá použít podruhé ani otestovat

## 2. Definice a volání

Příkaz `def` funkci **vytvoří a uloží pod jméno**. Nic se tím nespustí. Teprve
**volání** se závorkami tělo funkce provede.

In [3]:
def pozdrav():
    print("Telo funkce se provedlo.")


print("Definice sama nic nevypsala.")
pozdrav()
pozdrav()

Definice sama nic nevypsala.
Telo funkce se provedlo.
Telo funkce se provedlo.


Funkce je **hodnota** jako každá jiná, navázaná na jméno. Bez závorek se na ni
jen odkážeme, se závorkami ji spustíme.

In [4]:
print(odpor_vrstvy)              # hodnota: funkce
print(type(odpor_vrstvy))
print(odpor_vrstvy(0.30, 0.18))  # az zavorky ji spusti

<function odpor_vrstvy at 0x0000023925CEBE20>
<class 'function'>
1.6666666666666667


Proto je zapomenutá závorka zrádná chyba: program nespadne, jen se nic
nespočítá a v proměnné skončí funkce místo čísla.

In [5]:
vysledek = odpor_vrstvy          # chyba: jen druhe jmeno pro funkci
print(vysledek)

vysledek = odpor_vrstvy(0.30, 0.18)   # spravne
print(vysledek)

<function odpor_vrstvy at 0x0000023925CEBE20>
1.6666666666666667


Slovníkem z lekce 4: `def` je **příkaz**, volání funkce je **výraz** - má
hodnotu, takže se dá přiřadit nebo vložit do `print()`.

## 3. Parametry a argumenty

Dva pojmy, které se pletou:

* **parametr** je jméno v definici funkce
* **argument** je konkrétní hodnota při volání

Argumenty se dají předat dvěma způsoby a parametry mohou mít výchozí hodnotu.

In [6]:
def soucinitel_prostupu(R_konstrukce, R_si=0.13, R_se=0.04):
    return 1 / (R_si + R_konstrukce + R_se)


print(soucinitel_prostupu(5.90))                 # pozicne, zbytek vychozi
print(soucinitel_prostupu(5.90, 0.10))           # druhy argument pozicne
print(soucinitel_prostupu(5.90, R_se=0.10))      # pojmenovany argument
print(soucinitel_prostupu(R_konstrukce=5.90))    # pojmenovat lze i prvni

0.16474464579901152
0.16556291390728478
0.1631321370309951
0.16474464579901152


Pravidla:

* pozicní argumenty se přiřazují **v pořadí**, pojmenované **podle jména**
* pojmenované argumenty musí být až **za** pozicními
* v definici musí být parametry s výchozí hodnotou až **za** těmi bez ní
* pojmenovaný argument je zároveň dokumentace na místě volání: z
  `soucinitel_prostupu(5.90, R_se=0.10)` je poznat, co to `0.10` je

In [7]:
print(soucinitel_prostupu(R_si=0.10, 5.90))

SyntaxError: positional argument follows keyword argument (4245531256.py, line 1)

Python umí i funkce s **proměnným počtem argumentů** - v signatuře se zapisují
jako `*args` a `**kwargs`. Potkáte je v cizím kódu i v dokumentaci knihoven.
Souvisí to s tím, jak se kód skládá z modulů, takže si je necháme na lekci 7.

## 4. Signatura, anotace typů a docstring

**Signatura** je první řádek definice. Říká všechno, co potřebujete vědět,
abyste funkci zavolali:

```python
def soucinitel_prostupu(R_konstrukce: float, R_si: float = 0.13) -> float:
#   ^ jmeno             ^ parametr     ^ typ  ^ vychozi hodnota    ^ typ vysledku
```

**Anotace typů** (`: float`, `-> float`) jsou **popis**, ne příkaz. Python je
nevynucuje, stejně jako u `dataclass` v lekci 3 - funkci lze zavolat i s textem
a spadne to teprve na dělení. Čtenáři a editoru ale řeknou, s čím funkce počítá,
a VS Code podle nich napovídá.

**Docstring** je text v trojitých uvozovkách jako první věc v těle funkce.
Zobrazí ho `help()`, které znáte z lekce 2.

In [8]:
def soucinitel_prostupu(R_konstrukce: float, R_si: float = 0.13,
                        R_se: float = 0.04) -> float:
    """Vrati soucinitel prostupu tepla U ve W/(m2.K).

    R_konstrukce -- tepelny odpor konstrukce v m2.K/W
    R_si, R_se   -- odpory pri prestupu tepla na vnitrni a vnejsi strane
    """
    return 1 / (R_si + R_konstrukce + R_se)


print(soucinitel_prostupu(5.90))

0.16474464579901152


In [9]:
help(soucinitel_prostupu)

Help on function soucinitel_prostupu in module __main__:

soucinitel_prostupu(R_konstrukce: float, R_si: float = 0.13, R_se: float = 0.04) -> float
    Vrati soucinitel prostupu tepla U ve W/(m2.K).

    R_konstrukce -- tepelny odpor konstrukce v m2.K/W
    R_si, R_se   -- odpory pri prestupu tepla na vnitrni a vnejsi strane



## 5. `return`, `None` a víc návratových hodnot

`return` předá hodnotu zpět tomu, kdo funkci zavolal, a funkci tím **okamžitě
ukončí**. Co je za ním, se už neprovede.

In [10]:
def stav_prurezu(vyuziti):
    if vyuziti <= 1.0:
        return "VYHOVUJE"
    return "NEVYHOVUJE"      # sem se dostaneme jen kdyz podminka neplatila


print(stav_prurezu(0.86), stav_prurezu(1.09))

VYHOVUJE NEVYHOVUJE


Funkce bez `return` vrací `None`. Není to chyba - funkce, která má jen něco
vypsat, nemá co vracet. Chyba je na `return` zapomenout u funkce, od které
výsledek čekáme.

In [11]:
def vypis_stav(vyuziti):
    print(f"Vyuziti {vyuziti:.2f}")


vysledek = vypis_stav(0.86)
print(vysledek)              # funkce nic nevraci, tedy None

Vyuziti 0.86
None


Víc hodnot se vrací jako **tuple** a na druhé straně se rozbalí (lekce 3):

In [12]:
def rozmery_desky(delka, sirka, tloustka):
    objem = delka * sirka * tloustka
    hmotnost = objem * 2500 / 1000      # t
    cena = objem * 2900                 # Kc
    return objem, hmotnost, cena        # tuple tri hodnot


objem, hmotnost, cena = rozmery_desky(12.5, 8.0, 0.25)

print(f"{objem:.2f} m3, {hmotnost:.2f} t, {cena:.2f} Kč")

25.00 m3, 62.50 t, 72500.00 Kč


## 6. Co funkce vidí

Jména, která vzniknou v těle funkce, jsou **lokální**. Po skončení funkce
zmizí a zvenku nejsou vidět.

In [13]:
def spocti_odpor(d, lam):
    odpor = d / lam          # lokalni jmeno
    return odpor


print(spocti_odpor(0.30, 0.18))
print(odpor)                 # lokalni jmeno vne funkce neexistuje

1.6666666666666667


NameError: name 'odpor' is not defined

Vnější jména funkce **čte** bez problémů. Hodí se to na konstanty:

In [14]:
DPH = 0.21                   # konstanta vne funkce


def cena_s_dph(cena):
    return cena * (1 + DPH)  # vnejsi jmeno se jen cte


print(cena_s_dph(72500))

87725.0


Přiřazením v těle funkce ale vznikne **nové lokální jméno**. Vnější proměnná
zůstane nezměněná, i když se jmenuje stejně:

In [15]:
pocet_prvku = 10


def pridej_prvek():
    pocet_prvku = 99         # vytvori lokalni jmeno, vnejsi necha byt


pridej_prvek()
print(pocet_prvku)

10


Existuje klíčové slovo `global`, které tohle obejde. Nepoužívejte ho - funkce
má dostat vstupy argumenty a výsledek vrátit pomocí `return`. Kód, který mění
globální proměnné, se nedá přečíst ani otestovat.

## 7. Čistá funkce a vedlejší efekty

**Čistá funkce** (pure function) splňuje dvě věci:

* výsledek závisí **jen na argumentech** - stejné argumenty, vždy stejný výsledek
* **nic mimo sebe nemění**

**Vedlejší efekt** (side effect) je všechno ostatní: výpis na obrazovku, změna
seznamu z argumentu, zápis do globální proměnné nebo do souboru, čtení aktuálního
času. Vedlejší efekty nejsou zakázané - program, který nic nevypíše, je k ničemu.
Důležité je **vědět, které funkce je mají**.

In [16]:
def odpor_vrstvy(d, lam):    # cista: jen spocita a vrati
    return d / lam


def vypis_odpor(d, lam):     # vedlejsi efekt: vypisuje na obrazovku
    print(f"R = {d / lam:.2f}")


print(odpor_vrstvy(0.30, 0.18))
vypis_odpor(0.30, 0.18)

1.6666666666666667
R = 1.67


Nejzrádnější efekt je **změna měnitelného argumentu**. Z lekce 3 víme, že
přiřazení nekopíruje - funkce dostane tentýž seznam, který má volající:

In [17]:
def pridej_vymeru(vymery, nova):
    vymery.append(nova)      # meni seznam volajiciho!
    return vymery


puvodni = [25.0, 18.4]
vysledek = pridej_vymeru(puvodni, 7.2)

print("vysledek:", vysledek)
print("puvodni: ", puvodni)  # zmenil se taky

vysledek: [25.0, 18.4, 7.2]
puvodni:  [25.0, 18.4, 7.2]


Čistá varianta vrátí nový seznam a původní nechá být:

In [18]:
def s_pridanou_vymerou(vymery, nova):
    return vymery + [nova]   # novy seznam, original zustava


puvodni = [25.0, 18.4]
vysledek = s_pridanou_vymerou(puvodni, 7.2)

print("vysledek:", vysledek)
print("puvodni: ", puvodni)

vysledek: [25.0, 18.4, 7.2]
puvodni:  [25.0, 18.4]


### Past: měnitelná výchozí hodnota

Výchozí hodnota se vyhodnotí **jednou při definici**, ne při každém volání.
Měnitelný seznam jako výchozí hodnota si proto pamatuje, co se do něj přidalo
minule:

In [19]:
def pridej_vadu(popis, vady=[]):     # SPATNE
    vady.append(popis)
    return vady


print(pridej_vadu("trhlina v rohu"))
print(pridej_vadu("odlupujici se omitka"))   # pamatuje si predchozi vadu!

['trhlina v rohu']
['trhlina v rohu', 'odlupujici se omitka']


Správně se jako výchozí hodnota použije `None` a prázdný seznam se vyrobí
v těle funkce:

In [20]:
def pridej_vadu(popis, vady=None):   # SPRAVNE
    if vady is None:
        vady = []
    vady.append(popis)
    return vady


print(pridej_vadu("trhlina v rohu"))
print(pridej_vadu("odlupujici se omitka"))

['trhlina v rohu']
['odlupujici se omitka']


## 8. Funkce je hodnota

Z části 2 víme, že funkce je hodnota navázaná na jméno. Dá se tedy uložit do
jiné proměnné a hlavně **předat jako argument jiné funkci**.

In [21]:
def na_tuny(kg):
    return kg / 1000


prevod = na_tuny             # bez zavorek: druhe jmeno pro tutez funkci
print(prevod(62500))

62.5


Typické použití je `sorted()` s parametrem `key`. Ten čeká **funkci**, kterou
`sorted()` zavolá pro každý prvek, a tříděno je podle toho, co funkce vrátí.

In [22]:
prvky = [
    {"kod": "SO-03", "objem": 7.2},
    {"kod": "SO-01", "objem": 31.6},
    {"kod": "SO-02", "objem": 18.4},
]


def podle_objemu(prvek):
    return prvek["objem"]


setrideno = sorted(prvky, key=podle_objemu)
print([p["kod"] for p in setrideno])

['SO-03', 'SO-02', 'SO-01']


Pro takhle krátkou funkci se hodí `lambda` - funkce bez jména zapsaná na jednom
řádku. Používejte ji **jen tam, kde se hned použije**, typicky jako `key`.
Cokoli delšího napište normálním `def`.

In [23]:
print([p["kod"] for p in sorted(prvky, key=lambda p: p["objem"])])
print([p["kod"] for p in sorted(prvky, key=lambda p: p["kod"], reverse=True)])

['SO-03', 'SO-02', 'SO-01']
['SO-03', 'SO-02', 'SO-01']


Tím se dovysvětluje zápis z lekce 3. `cenik.get` je metoda předaná jako funkce
a `max()` ji zavolá pro každý klíč slovníku:

In [24]:
cenik = {"beton": 2900, "vyztuz": 32000, "bedneni": 450}

print(max(cenik, key=cenik.get))

vyztuz


## 9. Jedna funkce na celou kolekci

Funkci napsanou pro jednu hodnotu pustíme na celý seznam. Nejčastěji
comprehensionem z lekce 5:

In [25]:
tloustky = [0.30, 0.16, 0.015]
lambdy = [0.18, 0.038, 0.80]

odpory = [odpor_vrstvy(d, lam) for d, lam in zip(tloustky, lambdy)]

print([round(R, 2) for R in odpory])
print(f"Celkem {sum(odpory):.2f} m2.K/W")

[1.67, 4.21, 0.02]
Celkem 5.90 m2.K/W


Druhá možnost je funkce `map()`, která funkci aplikuje na každý prvek. Vrací
líný objekt, takže se výsledek obvykle převede na seznam:

In [26]:
hmotnosti_kg = [62500, 46000, 18000]

print(map(na_tuny, hmotnosti_kg))          # liny objekt
print(list(map(na_tuny, hmotnosti_kg)))    # az list ho rozbali

[62.5, 46.0, 18.0]


Comprehension a `map()` dělají totéž. Comprehension je čitelnější a zvládne
i podmínku, `map()` se hodí, když už funkci máte pojmenovanou. V lekci 8 uvidíte
NumPy, kde se funkce aplikuje na celé pole bez cyklu i bez `map()`.

## 10. Metody jsou taky funkce

Metoda `kod.upper()` vypadá jako něco jiného než funkce `len(kod)`, ale není.
Metoda je obyčejná funkce uložená u **typu** a objekt před tečkou se jí předá
jako **první argument**. Tomu prvnímu argumentu se v Pythonu říká `self`.

In [27]:
kod = "so-01"

print(kod.upper())         # obvykly zapis
print(str.upper(kod))      # totez: funkce s objektem jako prvnim argumentem

SO-01
SO-01


In [28]:
vymery = [25.0, 18.4]

print(vymery.append)       # metoda uz navazana na tento seznam
print(list.append)         # ta sama funkce, jeste bez objektu

<built-in method append of list object at 0x0000023925CA75C0>
<method 'append' of 'list' objects>


Praktický důsledek jsme použili v části 8: metodu lze předat jako argument
(`key=cenik.get`), protože je to hodnota stejně jako každá jiná funkce.

## 11. Časté chyby

**Zapomenutý `return`.** Funkce spočítá a výsledek zahodí. Vrátí `None`.

In [29]:
def objem_desky(delka, sirka, tloustka):
    delka * sirka * tloustka      # chybi return


print(objem_desky(12.5, 8.0, 0.25))

None


**Volání před definicí.** Soubor se vykonává odshora dolů, takže v okamžiku
volání funkce ještě nemusí existovat.

In [30]:
print(plocha_desky(12.5, 8.0))


def plocha_desky(delka, sirka):
    return delka * sirka

NameError: name 'plocha_desky' is not defined

**Špatný počet argumentů.** Chybová hláška řekne přesně, který parametr chybí.

In [31]:
print(odpor_vrstvy(0.30))

TypeError: odpor_vrstvy() missing 1 required positional argument: 'lam'

**Přepsání zabudované funkce.** Jména jako `sum`, `list`, `len`, `max` nebo
`type` nepoužívejte pro vlastní proměnné - zabudovaná funkce tím zmizí.

In [32]:
sum = 0                  # prepsali jsme zabudovanou funkci sum()
print(sum([1, 2, 3]))

TypeError: 'int' object is not callable

In [33]:
del sum                  # smazani vlastniho jmena zabudovanou funkci vrati
print(sum([1, 2, 3]))

6


## Shrnutí

| Zápis | Význam |
|---|---|
| `def jmeno(a, b=1):` | definice, nic se nespustí |
| `jmeno(5)` | volání, teprve teď se tělo provede |
| `jmeno` | funkce jako hodnota, bez spuštění |
| `d: float` a `-> float` | anotace typů, popis bez vynucení |
| `return a, b` | vrácení dvou hodnot jako tuple |
| `key=lambda p: p["objem"]` | funkce předaná jako argument |
| `list(map(f, seznam))` | funkce aplikovaná na celou kolekci |

Co si odnést:

* `def` funkci vytvoří, teprve závorky ji spustí
* parametr je v definici, argument při volání; pojmenovaný argument je
  i dokumentace
* funkce bez `return` vrací `None`
* přiřazení v těle funkce vytvoří lokální jméno, vnější proměnnou nezmění
* čistá funkce závisí jen na argumentech a nic nemění; změna seznamu
  z argumentu je vedlejší efekt, který volající nečeká
* jako výchozí hodnotu nikdy nedávejte měnitelnou kolekci, použijte `None`
* funkce je hodnota, dá se předat jako argument
* metoda je funkce, jejímž prvním argumentem je sám objekt

V lekci 7 funkce uložíme do vlastních modulů a ukážeme si `*args` a `**kwargs`.